# 2a — Preprocessing: yellow taxi trip records

**Input:** `data/landing/tlc/yellow_tripdata_YYYY-MM.parquet` (18 files, Jan 2023 – Jun 2024, 58,642,319 rows)

**Outputs:**

| File | Contents |
|---|---|
| `data/raw/trips_clean.parquet` | Full citywide cleaned trip records, partitioned by month. Consumed by notebook 3 (distribution and outlier analysis). |
| `data/curated/taxi_airport_hourly.parquet` | Table A — one row per `(pickup_date, pickup_hour, airport)` for JFK and LGA. 26,256 rows. |
| `data/curated/preprocessing_counts_taxi.csv` | Record count after each filter, for the preprocessing table in the report. |
| `data/curated/shapes_taxi.json` | Headline shapes and retention figures quoted in the report. |

## Order of operations, and why

Cleaning is applied to **all 58.6M rows before** the airport subset is taken, not after.
The subject specification requires the full distribution to be used when analysing
distributions, aggregating attributes, and performing outlier analysis. Filtering to two
taxi zones first would be cheaper, but the resulting outlier analysis would describe only
airport trips. Airport trips have an atypical fare distribution — the JFK flat fare
compresses a large mass of records onto a single value — so characterising them against
the citywide baseline is the more informative comparison, and it is the one the
specification asks for.

## Removal policy: three tiers

1. A record is **removed** only if it violates a documented business rule or is
   logically impossible.
2. A record that is merely **extreme** is retained and characterised in the outlier
   analysis of notebook 3. Discarding extreme-but-possible records here would pre-empt
   that analysis and quietly narrow the distribution being reported.
3. A record whose **metadata is unrecorded** — a missing rate code, a missing passenger
   count — is retained and *flagged*. The trip happened; only a field describing it is
   absent. Deleting these would be the most damaging of the three mistakes, because
   `n_pickups` is the response variable and the missing fields are not distributed
   evenly across vendors, hours, or airports. Every statistic derived from such a field
   therefore carries its own denominator column rather than silently borrowing
   `n_pickups`.

In [1]:
"""Preprocessing of the TLC yellow taxi trip records.

Reads the raw monthly parquet files, harmonises their schemas, applies
business-rule filters to the full citywide dataset, and aggregates the airport
subset to hourly counts on the grid used by every downstream notebook.
"""

import json
import sys
from functools import reduce
from operator import and_
from pathlib import Path

from pyspark.sql import functions as F

# Session configuration, schema normalisation, the verified zone identifiers,
# and the reporting helpers shared with notebook 2b live in
# `scripts/spark_utils.py` so that every notebook loads the data identically
# and the logic can be linted as ordinary source.
sys.path.append(str(Path("..") / "scripts"))
from spark_utils import (  # noqa: E402
    ARRIVAL_AIRPORTS,
    JFK_ZONE,
    LGA_ZONE,
    count_waterfall,
    create_spark_session,
    hour_spine,
    load_trips,
    print_waterfall,
)

# --- Paths -----------------------------------------------------------------
# Notebook is expected to run from `notebooks/`.
PROJECT_ROOT = Path("..").resolve()
LANDING_DIR = PROJECT_ROOT / "data" / "landing" / "tlc"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CURATED_DIR = PROJECT_ROOT / "data" / "curated"

for directory in (RAW_DIR, CURATED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# --- Study window ----------------------------------------------------------
# Inclusive of the start date, exclusive of the end date.
WINDOW_START = "2023-01-01"
WINDOW_END = "2024-07-01"
WINDOW_DAYS = 547
EXPECTED_ROWS = WINDOW_DAYS * 24 * len(ARRIVAL_AIRPORTS)  # 26,256

# --- Taxi zones ------------------------------------------------------------
AIRPORT_ZONES = {JFK_ZONE: "JFK", LGA_ZONE: "LGA"}
# EWR (zone 1) is deliberately excluded. Yellow taxis may drop off at Newark but
# may not pick up there, so EWR enters this study only through the flight-side
# features built in notebook 2b.

# The airport codes must agree with the ones notebook 2b keys its flight table
# on, or the join in 2c silently drops rows instead of failing.
assert tuple(AIRPORT_ZONES.values()) == ARRIVAL_AIRPORTS

In [2]:
spark = create_spark_session(app_name="MAST30034 — taxi preprocessing")
spark.sparkContext.setLogLevel("WARN")
spark.version

your 131072x1 screen size is bogus. expect trouble
26/08/16 19:00:53 WARN Utils: Your hostname, iphone resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/16 19:00:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/16 19:00:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.1'

## Step 1 — Load and harmonise the monthly files

`load_trips` reads every `yellow_tripdata_*.parquet` file in the landing directory,
passes each through `normalise_schema`, and unions them with `unionByName` so that a
difference in column ordering cannot silently misalign the frames.

`normalise_schema` handles the two schema differences across the eighteen months: the
airport surcharge is named `airport_fee` in some files and `Airport_fee` in others, and
the identifier and count columns vary between 64-bit and 32-bit integers and doubles.
Both are documented in the README.

In [3]:
trips_raw = load_trips(spark, LANDING_DIR)
print(f"{len(trips_raw.columns)} columns after normalisation")

19 columns after normalisation


In [4]:
# Derived quantities used by the filters below. Both are computed before any
# filtering so that they are available to the count waterfall in Step 3.
trips_raw = (
    trips_raw
    .withColumn(
        "trip_duration_s",
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime"),
    )
    .withColumn(
        "implied_mph",
        F.when(
            F.col("trip_duration_s") > 0,
            F.col("trip_distance") / (F.col("trip_duration_s") / 3600.0),
        ),
    )
)

## Step 2 — Filter definitions

Each filter below is stated with the rule it enforces. Thresholds that depend on the TLC
fare schedule are held in named constants so that the value used is visible in one place
and can be checked against the published schedule.

| # | Filter | Rule being enforced |
|---|---|---|
| 1 | Pickup inside the study window | Each monthly file contains a small number of records timestamped outside its own month, arising from meter and upload errors. Retaining them would misstate the timeline. |
| 2 | Drop-off after pickup | A trip cannot end before it starts. |
| 3 | Duration between 1 minute and 6 hours | Sub-minute records are meter mis-taps rather than journeys; multi-hour records are meters left running after the passenger has left. |
| 4 | Positive trip distance | A recorded distance of zero means no journey took place. |
| 5 | Positive fare and total | Negative amounts are voided transactions and chargebacks, not trips. |
| 6 | Metered fare at or above the initial charge | The initial unit charge on Rate Code 1 has been $3.00 since 19 December 2022, so no standard-rate trip in this window can bill less. Applies to Rate Code 1 only, since the flat and negotiated rates are priced differently. |
| 7 | Implied speed under 80 mph | Not physically attainable across the five boroughs; indicates a corrupt odometer or timestamp reading. |
| 8 | Documented rate code | The data dictionary defines codes 1–6 and 99 = Null/unknown. Any other value is undefined and is removed. |
| 9 | Documented payment type | The data dictionary defines codes 0–6, where 0 = Flex Fare. Any other value is undefined and is removed. |

Rules that are recorded but **not** enforced as removals, because they do not invalidate
a record:

- `tip_amount` is populated for card payments only; cash tips are not recorded. Any tip
  statistic must therefore be computed over `payment_type = 1` alone. This is handled in
  the aggregation rather than by deleting cash trips, which are valid trips.
- Extreme-but-possible values — a 90-mile Rate Code 4 run to Westchester, a $200 fare —
  are retained for the outlier analysis.
- A rate code that is **unrecorded** is retained. The dictionary defines `99` as
  Null/unknown, so a literal null and a `99` carry the same meaning: the code was not
  captured. Neither invalidates the trip — the meter ran, the passenger was carried — so
  both are retained and flagged as `ratecode_missing`. Note that `.isin()` alone would
  delete the nulls silently, because the predicate evaluates to null and `.where()` drops
  null predicates; the filters below admit nulls explicitly.
- A **passenger count** that is unrecorded is likewise retained and flagged. It arrives
  null in the same records as the rate code.
- **Flex Fare** trips (`payment_type = 0`) are retained. The code is documented, and the
  trips are ordinary yellow taxi journeys paid for through an app at an upfront price.

These decisions matter more than their row counts suggest. `n_pickups` is the response
variable, and none of these groups is distributed evenly across hours, airports, or
vendors: Flex Fare is app-dispatched, and unrecorded fields are concentrated in the
vendors that do not populate them. Deleting any of them would push a vendor's reporting
behaviour, or a fare programme's market share, into the target where it would be read as
demand. Step 4a quantifies exactly who the retained records belong to.

In [5]:
# Initial unit charge on the standard city rate (Rate Code 1). Raised from
# $2.50 to $3.00 effective 19 December 2022 by TLC Industry Notice #22-02, so
# the higher figure applies across the whole of this study window. Cite the
# notice, or the TLC passenger fare page, when this threshold appears in the
# report.
MIN_METERED_FARE = 3.00

MAX_IMPLIED_MPH = 80.0
MIN_DURATION_S = 60
MAX_DURATION_S = 6 * 3600

# Codes defined by the data dictionary (yellow taxi, 18 March 2025 revision).
# Rate code 99 is the dictionary's own "Null/unknown" sentinel and payment
# type 0 is a Flex Fare trip; both are documented, so both are retained. Only
# values the dictionary does not define at all are removed here.
RATECODE_STANDARD = 1
RATECODE_JFK_FLAT = 2
RATECODE_UNKNOWN = 99
DOCUMENTED_RATECODES = [1, 2, 3, 4, 5, 6, RATECODE_UNKNOWN]
DOCUMENTED_PAYMENT_TYPES = [0, 1, 2, 3, 4, 5, 6]
PAYMENT_CARD = 1
PAYMENT_FLEX_FARE = 0

# Ordered list of (label, predicate). The label is reproduced verbatim in the
# preprocessing table of the report, so keep the two in step.
FILTERS = [
    (
        "Pickup within study window",
        (F.col("tpep_pickup_datetime") >= F.lit(WINDOW_START).cast("timestamp"))
        & (F.col("tpep_pickup_datetime") < F.lit(WINDOW_END).cast("timestamp")),
    ),
    (
        "Drop-off after pickup",
        F.col("trip_duration_s") > 0,
    ),
    (
        "Duration between 1 min and 6 h",
        (F.col("trip_duration_s") >= MIN_DURATION_S)
        & (F.col("trip_duration_s") <= MAX_DURATION_S),
    ),
    (
        "Positive trip distance",
        F.col("trip_distance") > 0,
    ),
    (
        "Positive fare and total",
        (F.col("fare_amount") > 0) & (F.col("total_amount") > 0),
    ),
    (
        "Metered fare at or above initial charge",
        # `eqNullSafe` rather than `==` so that a missing rate code yields
        # False rather than null here: `null != 1` is null, which `.where()`
        # would drop, deleting the very records this filter does not judge.
        ~F.col("ratecodeid").eqNullSafe(RATECODE_STANDARD)
        | (F.col("fare_amount") >= MIN_METERED_FARE),
    ),
    (
        "Implied speed below 80 mph",
        F.col("implied_mph") < MAX_IMPLIED_MPH,
    ),
    (
        "Documented rate code",
        F.col("ratecodeid").isNull()
        | F.col("ratecodeid").isin(DOCUMENTED_RATECODES),
    ),
    (
        "Documented payment type",
        F.col("payment_type").isNull()
        | F.col("payment_type").isin(DOCUMENTED_PAYMENT_TYPES),
    ),
]

## Step 3 — Record counts after each filter

The specification requires the dataset shape to be stated at each filtering step, and the
figures are cross-checked against this code during marking.

Calling `.count()` after each of the nine filters would trigger nine full scans of 58.6M
rows. `count_waterfall` instead evaluates the cumulative conjunction of the predicates in
a **single pass**: summing `predicate_1`, then `predicate_1 AND predicate_2`, and so on,
gives exactly the sequential post-filter counts. It lives in `scripts/spark_utils.py`
because notebook 2b needs the same table for the flight data, and two copies of a
counting routine is two chances for the two preprocessing tables to disagree.

It uses `F.when(condition, 1).otherwise(0)` rather than casting the boolean, so that a
null predicate counts as an exclusion — matching the behaviour of `.where()`, which also
drops null predicates. That equivalence is the reason the filters above have to admit
nulls explicitly wherever a null is meant to survive.

In [6]:
waterfall = count_waterfall(trips_raw, FILTERS)
print_waterfall(waterfall)

Raw ingest                                   58,642,319  removed          —  (100.00% of raw)
Pickup within study window                   58,642,192  removed        127  (100.00% of raw)
Drop-off after pickup                        58,620,446  removed     21,746  ( 99.96% of raw)
Duration between 1 min and 6 h               57,922,347  removed    698,099  ( 98.77% of raw)
Positive trip distance                       57,243,639  removed    678,708  ( 97.61% of raw)
Positive fare and total                      56,642,200  removed    601,439  ( 96.59% of raw)
Metered fare at or above initial charge      56,641,660  removed        540  ( 96.59% of raw)
Implied speed below 80 mph                   56,637,949  removed      3,711  ( 96.58% of raw)
Documented rate code                         56,637,949  removed          0  ( 96.58% of raw)
Documented payment type                      56,637,949  removed          0  ( 96.58% of raw)


In [7]:
# Persist for the report table. Retention is reported against the raw ingest.
raw_total = waterfall[0][1]
rows = [
    {
        "step": label,
        "rows": remaining,
        "removed": removed,
        "pct_of_raw": round(100 * remaining / raw_total, 3),
    }
    for label, remaining, removed in waterfall
]

counts_path = CURATED_DIR / "preprocessing_counts_taxi.csv"
spark.createDataFrame(rows).coalesce(1).toPandas().to_csv(counts_path, index=False)
print(f"Written to {counts_path}")

Written to /home/tavish/projects/project-1-individual-SavvyHack/data/curated/preprocessing_counts_taxi.csv


## Step 4 — Apply the filters and persist the cleaned citywide dataset

This is the dataset that notebook 3 uses for the distribution and outlier analysis. It is
written before the airport subset is taken, which is the point of the ordering decision
made at the top of this notebook.

It is partitioned by month so that the analysis notebook can read a subset without a full
scan, and so that any month-specific data quality problem is isolated to one partition.

In [8]:
trips_clean = (
    trips_raw
    .where(reduce(and_, (predicate for _, predicate in FILTERS)))
    .withColumn("pickup_month", F.date_format("tpep_pickup_datetime", "yyyy-MM"))
    # Retained, but marked, so that any statistic derived from these columns
    # can state its own denominator rather than assume one. A null and a 99
    # both mean "not recorded" and are flagged identically.
    .withColumn(
        "ratecode_missing",
        F.col("ratecodeid").isNull()
        | (F.col("ratecodeid") == RATECODE_UNKNOWN),
    )
    .withColumn("passenger_count_missing", F.col("passenger_count").isNull())
    .withColumn("payment_type_missing", F.col("payment_type").isNull())
    .withColumn(
        "flex_fare",
        F.coalesce(F.col("payment_type") == PAYMENT_FLEX_FARE, F.lit(False)),
    )
)

(
    trips_clean
    .write
    .mode("overwrite")
    .partitionBy("pickup_month")
    .parquet(str(RAW_DIR / "trips_clean.parquet"))
)
print("Cleaned citywide dataset written.")

26/08/16 19:01:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/08/16 19:01:29 WARN DAGScheduler: Broadcasting large task binary with size 1015.8 KiB


Cleaned citywide dataset written.


In [9]:
# Read back from disk rather than recomputing the filter chain for every
# downstream action. Without this, each subsequent stage re-reads and re-filters
# all 18 monthly files.
trips_clean = spark.read.parquet(str(RAW_DIR / "trips_clean.parquet"))
clean_total = trips_clean.count()
assert clean_total == waterfall[-1][1], (
    f"Written {clean_total:,} rows but waterfall predicted {waterfall[-1][1]:,}"
)
print(f"{clean_total:,} rows verified on disk")

56,637,949 rows verified on disk


### Step 4a — Who are the records with unrecorded metadata?

Retaining roughly three million records on the argument that deleting them would bias the
target is only sound if it is actually checked, so the block is profiled here rather than
asserted about. Two questions decide how it is described in the report:

1. **Is it one vendor?** If the missing fields concentrate in a single `vendorid`, the
   pattern is a reporting convention rather than a data corruption, and deleting it would
   have removed that vendor's market share from the target.
2. **Where are the pickups?** If the pickup zone is `264` (Unknown) or `265` (Outside of
   NYC), the location is not trustworthy either, and the airport counts would be
   under-stated in a way that retention does *not* fix. If the zones are ordinary, the
   airport share of the block is a real fact about where that vendor operates.

The second question is the one that matters for this study, because the whole analysis is
keyed on pickup zone.

In [10]:
UNKNOWN_ZONES = [264, 265]  # "Unknown" and "Outside of NYC / N.A." in the lookup

missing_profile = (
    trips_clean
    .where(F.col("ratecode_missing"))
    .groupBy(
        "vendorid",
        F.when(F.col("pulocationid").isin(list(AIRPORT_ZONES)), "airport zone")
        .when(F.col("pulocationid").isin(UNKNOWN_ZONES), "unknown zone")
        .otherwise("other NYC zone")
        .alias("zone_group"),
    )
    .agg(
        F.count("*").alias("trips"),
        F.avg(F.col("passenger_count").isNull().cast("double"))
        .alias("share_passenger_null"),
        F.avg(F.col("flex_fare").cast("double")).alias("share_flex_fare"),
    )
    .orderBy(F.desc("trips"))
)
missing_profile.show(truncate=False)

missing_total = trips_clean.where(F.col("ratecode_missing")).count()
passenger_missing_total = trips_clean.where(F.col("passenger_count_missing")).count()
print(f"Unrecorded rate code:      {missing_total:,} "
      f"({100 * missing_total / clean_total:.2f}% of cleaned trips)")
print(f"Unrecorded passenger count: {passenger_missing_total:,} "
      f"({100 * passenger_missing_total / clean_total:.2f}% of cleaned trips)")

+--------+--------------+-------+--------------------+------------------+
|vendorid|zone_group    |trips  |share_passenger_null|share_flex_fare   |
+--------+--------------+-------+--------------------+------------------+
|2       |other NYC zone|2201776|1.0                 |1.0               |
|1       |other NYC zone|959434 |0.590909848931766   |0.590909848931766 |
|1       |airport zone  |15574  |0.9976884551175035  |0.9976884551175035|
|6       |unknown zone  |6104   |1.0                 |1.0               |
|1       |unknown zone  |2618   |0.806340718105424   |0.806340718105424 |
|2       |unknown zone  |1051   |1.0                 |1.0               |
|2       |airport zone  |370    |1.0                 |1.0               |
+--------+--------------+-------+--------------------+------------------+



Unrecorded rate code:      3,186,927 (5.63% of cleaned trips)
Unrecorded passenger count: 2,793,889 (4.93% of cleaned trips)


## Step 5 — Airport subset

Pickups at JFK (zone 132) and LaGuardia (zone 138) are extracted. Only the **pickup**
zone is used: the research question concerns whether a driver should join an airport
queue, so a trip *to* the airport is not part of the target.

### Independent validation of the zone filter

Rate Code 2 denotes the JFK flat fare, and is recorded by the meter independently of the
GPS-derived `PULocationID`. Almost every Rate Code 2 trip should therefore touch zone 132
at one end. Agreement between the two fields is evidence that the zone filter is
selecting what it is meant to select; a large disagreement would indicate that the zone
identifiers are wrong.

In [11]:
# Cross-check: what share of Rate Code 2 trips touch zone 132 at either end?
ratecode_2 = trips_clean.where(F.col("ratecodeid") == RATECODE_JFK_FLAT)
validation = ratecode_2.agg(
    F.count("*").alias("ratecode_2_trips"),
    F.avg(
        F.when(
            (F.col("pulocationid") == JFK_ZONE)
            | (F.col("dolocationid") == JFK_ZONE),
            1.0,
        ).otherwise(0.0)
    ).alias("share_touching_zone_132"),
).collect()[0]

print(f"Rate Code 2 trips:            {validation['ratecode_2_trips']:,}")
print(f"Share touching zone 132:      {validation['share_touching_zone_132']:.4f}")

Rate Code 2 trips:            2,019,422
Share touching zone 132:      0.9619


In [12]:
airport_trips = (
    trips_clean
    .where(F.col("pulocationid").isin(list(AIRPORT_ZONES)))
    .withColumn(
        "airport",
        F.when(F.col("pulocationid") == JFK_ZONE, F.lit("JFK"))
        .otherwise(F.lit("LGA")),
    )
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    # Read four more times below: the aggregation, the two totals used in the
    # validation checks, and the per-airport breakdown.
    .cache()
)

airport_total = airport_trips.count()
print(f"Airport pickups: {airport_total:,} "
      f"({100 * airport_total / clean_total:.2f}% of cleaned trips)")
airport_trips.groupBy("airport").count().show()

Airport pickups: 4,653,181 (8.22% of cleaned trips)
+-------+-------+
|airport|  count|
+-------+-------+
|    JFK|2753043|
|    LGA|1900138|
+-------+-------+



## Step 6 — Aggregate to Table A

One row per `(pickup_date, pickup_hour, airport)`.

`n_pickups` is the modelling target. The remaining columns describe the trips that
occurred in that hour and are used for the descriptive analysis and to construct
**lagged** features later. They must not be used as same-hour model inputs: the number of
card payments in hour *h* is not knowable to a driver deciding whether to join the queue
for hour *h*.

**Denominators are carried, not assumed.** Three columns are computed over a subset of
the hour's trips, and each is paired with a count so that the report never has to infer
the denominator and the modelling notebook cannot mistake a null for a zero:

| Statistic | Computed over | Denominator column |
|---|---|---|
| `share_flat_fare` | trips whose rate code was recorded | `n_ratecode_known` |
| `mean_passengers` | trips whose passenger count was recorded | `n_passenger_known` |
| `mean_tip_ratio` | card payments only, since cash tips are not recorded | `n_card_trips` |

**Revenue totals are carried as sums.** `sum_fare_amount`, `sum_total_amount`, and
`sum_tip_amount` cost nothing to compute here and make an earnings-shaped target
available to notebook 4 — total revenue per airport-hour rather than trip count — without
a second pass over 58.6M rows. A high-volume hour of short Astoria runs and a thin hour of
Manhattan fares are the same number of pickups and very different money, and which of the
two the model should predict is a decision better made with both quantities already on
disk. The means are retained alongside because the descriptive analysis reads more
naturally from them, and `median_fare` is not recoverable from a sum.

In [13]:
hourly = airport_trips.groupBy("pickup_date", "pickup_hour", "airport").agg(
    F.count("*").alias("n_pickups"),

    # --- Fare and revenue --------------------------------------------------
    F.sum("fare_amount").alias("sum_fare_amount"),
    F.sum("total_amount").alias("sum_total_amount"),
    F.avg("fare_amount").alias("mean_fare"),
    F.expr("percentile_approx(fare_amount, 0.5)").alias("median_fare"),
    F.avg("total_amount").alias("mean_total"),

    # --- Trip shape --------------------------------------------------------
    F.avg("trip_distance").alias("mean_distance_mi"),
    F.avg(F.col("trip_duration_s") / 60.0).alias("mean_duration_min"),

    # --- Passenger count, over the trips where it was recorded -------------
    F.sum(F.when(~F.col("passenger_count_missing"), 1).otherwise(0))
    .alias("n_passenger_known"),
    F.avg("passenger_count").alias("mean_passengers"),

    # --- Rate code, over the trips where it was recorded -------------------
    # Share of pickups on the JFK flat fare, a proxy for Manhattan-bound
    # demand. The third branch is left unset so that a missing rate code
    # contributes to neither the numerator nor the denominator, rather than
    # being counted as "not flat".
    F.sum(F.when(~F.col("ratecode_missing"), 1).otherwise(0))
    .alias("n_ratecode_known"),
    F.avg(
        F.when(F.col("ratecodeid") == RATECODE_JFK_FLAT, 1.0)
        .when(~F.col("ratecode_missing"), 0.0)
    ).alias("share_flat_fare"),

    # --- Payment -----------------------------------------------------------
    # Flex Fare is app-dispatched and priced upfront, so its share of an hour
    # describes how that hour's demand arrived, not just how it paid.
    F.avg(F.col("flex_fare").cast("double")).alias("share_flex_fare"),
    F.sum(F.when(F.col("payment_type") == PAYMENT_CARD, 1).otherwise(0))
    .alias("n_card_trips"),
    # Tips are recorded for card payments only (data dictionary), so both the
    # total and the mean are taken over card trips alone rather than over all
    # trips.
    F.sum(
        F.when(F.col("payment_type") == PAYMENT_CARD, F.col("tip_amount"))
    ).alias("sum_tip_amount"),
    F.avg(
        F.when(
            F.col("payment_type") == PAYMENT_CARD,
            F.col("tip_amount") / F.col("fare_amount"),
        )
    ).alias("mean_tip_ratio"),
)

non_empty_hours = hourly.count()
print(f"{non_empty_hours:,} non-empty airport-hours")

25,076 non-empty airport-hours


### Hours with no pickups

An airport-hour with zero pickups produces no group, so it is absent from the
aggregation above. Dropping those rows would remove precisely the low-demand hours the
model needs to learn — a count model fitted only to hours that had trips will
systematically overpredict the overnight period.

A complete spine of every `(date, hour, airport)` combination is therefore constructed by
`hour_spine` — the same helper notebook 2b uses, so the two tables are guaranteed to be
built on identical grids — and left-joined.

**Counts and sums are filled with zero; means and shares are left null.** An hour with no
trips genuinely earned nothing and carried nobody, so a zero is the correct revenue. But
the mean fare of no trips is undefined rather than zero, and filling it would inject a
spurious cluster at the origin into every fare plot and tell the model that empty hours
were cheap ones.

In [14]:
spine = hour_spine(
    spark,
    WINDOW_START,
    WINDOW_END,
    airports=ARRIVAL_AIRPORTS,
    date_col="pickup_date",
    hour_col="pickup_hour",
)
assert spine.count() == EXPECTED_ROWS, "Spine does not cover the study window"

# Counts of things that did not happen, and money that was not earned, are
# zero. Everything else is a statistic of an empty set and stays null.
ZERO_FILL = [
    "n_pickups",
    "n_card_trips",
    "n_ratecode_known",
    "n_passenger_known",
    "sum_fare_amount",
    "sum_total_amount",
    "sum_tip_amount",
]

table_a = (
    spine
    .join(hourly, ["pickup_date", "pickup_hour", "airport"], how="left")
    .fillna(0, subset=ZERO_FILL)
)

### Daylight saving

Timestamps are New York wall-clock time, so the spine contains two kinds of anomalous
hour:

- **Spring forward** (2023-03-12, 2024-03-10): the 02:00 hour does not exist. The spine
  creates a row for it and the left join fills it with zero pickups, which is an artefact
  rather than an observation of no demand.
- **Fall back** (2023-11-05): the 01:00 hour occurs twice, so that row contains two
  hours' worth of trips and will read as an outlier.

Six rows out of 26,256 are affected. They are flagged rather than deleted so that the
modelling notebook can exclude them explicitly and the report can state that it did.

In [15]:
DST_SPRING_FORWARD = ["2023-03-12", "2024-03-10"]  # 02:00 does not exist
DST_FALL_BACK = ["2023-11-05"]                      # 01:00 occurs twice

table_a = table_a.withColumn(
    "dst_anomaly",
    (
        F.col("pickup_date").cast("string").isin(DST_SPRING_FORWARD)
        & (F.col("pickup_hour") == 2)
    )
    | (
        F.col("pickup_date").cast("string").isin(DST_FALL_BACK)
        & (F.col("pickup_hour") == 1)
    ),
).cache()

table_a.where(F.col("dst_anomaly")).select(
    "pickup_date", "pickup_hour", "airport", "n_pickups"
).orderBy("pickup_date", "airport").show()

+-----------+-----------+-------+---------+
|pickup_date|pickup_hour|airport|n_pickups|
+-----------+-----------+-------+---------+
| 2023-03-12|          2|    JFK|        0|
| 2023-03-12|          2|    LGA|        0|
| 2023-11-05|          1|    JFK|       47|
| 2023-11-05|          1|    LGA|        0|
| 2024-03-10|          2|    JFK|        0|
| 2024-03-10|          2|    LGA|        0|
+-----------+-----------+-------+---------+



## Step 7 — Validate and write

Six checks before Table A leaves this notebook. Each one catches a failure that would
otherwise surface as a plausible-looking but wrong number much later — and checks 3 to 5
are the ones that would catch a denominator quietly reverting to `n_pickups`.

In [16]:
# 1. Every airport-hour in the window is present exactly once.
assert table_a.count() == EXPECTED_ROWS
assert table_a.dropDuplicates(
    ["pickup_date", "pickup_hour", "airport"]
).count() == EXPECTED_ROWS

# 2. No trip was lost or duplicated by the grouping and the join.
assert table_a.agg(F.sum("n_pickups")).collect()[0][0] == airport_total

# 3. The flat-fare share is defined in exactly the hours where a rate code was
#    observed, and nowhere else.
mismatched = table_a.where(
    F.col("share_flat_fare").isNull() != (F.col("n_ratecode_known") == 0)
).count()
assert mismatched == 0, f"{mismatched} rows disagree on the rate code denominator"

# 4. The same, for the passenger count.
mismatched = table_a.where(
    F.col("mean_passengers").isNull() != (F.col("n_passenger_known") == 0)
).count()
assert mismatched == 0, f"{mismatched} rows disagree on the passenger denominator"

# 5. Revenue survived the aggregation and the zero-fill. Compared with a
#    tolerance rather than for equality, because a distributed sum of doubles
#    is not associative and may differ in the last cents between two plans.
revenue_table = table_a.agg(F.sum("sum_total_amount")).collect()[0][0]
revenue_trips = airport_trips.agg(F.sum("total_amount")).collect()[0][0]
assert abs(revenue_table - revenue_trips) < 1.0, (
    f"Revenue disagrees: {revenue_table:,.2f} vs {revenue_trips:,.2f}"
)
print(f"Airport revenue reconciled: ${revenue_table:,.2f}")

# 6. Report how many genuine zero-demand hours exist, excluding the DST artefacts.
zero_hours = table_a.where(
    (F.col("n_pickups") == 0) & (~F.col("dst_anomaly"))
).count()
print(f"Genuine zero-pickup airport-hours: {zero_hours} of {EXPECTED_ROWS}")
print("All checks passed.")

Airport revenue reconciled: $354,004,177.23
Genuine zero-pickup airport-hours: 1175 of 26256
All checks passed.


In [17]:
(
    table_a
    .orderBy("pickup_date", "pickup_hour", "airport")
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(CURATED_DIR / "taxi_airport_hourly.parquet"))
)

table_a.orderBy("pickup_date", "pickup_hour", "airport").show(6, truncate=False)

+-----------+-----------+-------+---------+------------------+-----------------+------------------+-----------+-----------------+------------------+------------------+-----------------+------------------+----------------+-------------------+--------------------+------------+------------------+-------------------+-----------+
|pickup_date|pickup_hour|airport|n_pickups|sum_fare_amount   |sum_total_amount |mean_fare         |median_fare|mean_total       |mean_distance_mi  |mean_duration_min |n_passenger_known|mean_passengers   |n_ratecode_known|share_flat_fare    |share_flex_fare     |n_card_trips|sum_tip_amount    |mean_tip_ratio     |dst_anomaly|
+-----------+-----------+-------+---------+------------------+-----------------+------------------+-----------+-----------------+------------------+------------------+-----------------+------------------+----------------+-------------------+--------------------+------------+------------------+-------------------+-----------+
|2023-01-01 |0     

In [18]:
# Record the airport-side shapes alongside the citywide waterfall so that the
# preprocessing table in the report is complete. Retained-but-flagged records
# are reported at both scales: the citywide figure justifies the retention
# decision, the airport figure bounds its effect on the target.
airport_ratecode_known = table_a.agg(F.sum("n_ratecode_known")).collect()[0][0]
airport_passenger_known = table_a.agg(F.sum("n_passenger_known")).collect()[0][0]

shapes = {
    "raw_ingest_trips": raw_total,
    "cleaned_citywide_trips": clean_total,
    "removed_trips": raw_total - clean_total,
    "removed_pct": round(100 * (raw_total - clean_total) / raw_total, 3),
    "airport_pickups": airport_total,
    "airport_pickup_pct_of_clean": round(100 * airport_total / clean_total, 3),
    "non_empty_airport_hours": non_empty_hours,
    "table_a_rows": EXPECTED_ROWS,
    "genuine_zero_pickup_hours": zero_hours,
    # Retained rather than deleted; reported so the report can say how many.
    "citywide_missing_ratecode": missing_total,
    "citywide_missing_passenger_count": passenger_missing_total,
    "airport_missing_ratecode": airport_total - airport_ratecode_known,
    "airport_missing_passenger_count": airport_total - airport_passenger_known,
    "ratecode_2_trips": validation["ratecode_2_trips"],
    "ratecode_2_share_touching_jfk": round(
        float(validation["share_touching_zone_132"]), 4
    ),
}
with open(CURATED_DIR / "shapes_taxi.json", "w") as handle:
    json.dump(shapes, handle, indent=2)

shapes

{'raw_ingest_trips': 58642319,
 'cleaned_citywide_trips': 56637949,
 'removed_trips': 2004370,
 'removed_pct': 3.418,
 'airport_pickups': 4653181,
 'airport_pickup_pct_of_clean': 8.216,
 'non_empty_airport_hours': 25076,
 'table_a_rows': 26256,
 'genuine_zero_pickup_hours': 1175,
 'citywide_missing_ratecode': 3186927,
 'citywide_missing_passenger_count': 2793889,
 'airport_missing_ratecode': 15944,
 'airport_missing_passenger_count': 15908,
 'ratecode_2_trips': 2019422,
 'ratecode_2_share_touching_jfk': 0.9619}

In [19]:
spark.stop()

## What to carry into the report

- The count waterfall printed in Step 3 populates the preprocessing table. Quote the
  labels from `FILTERS` verbatim so the table and the code agree.
- The Rate Code 2 agreement figure from Step 5 justifies the airport zone identifiers
  without asking the reader to take them on trust.
- The share of cleaned trips that are airport pickups justifies the narrowing from
  citywide to airport, and the fact that the citywide dataset was cleaned first justifies
  the outlier analysis in notebook 3.
- The DST and zero-demand-hour handling are each worth one sentence in the preprocessing
  section. They are small, but they are the kind of decision a marker checks the code for.
- Rate code 99 and Flex Fare payments are retained, on the dictionary's own definitions.
  Cite the dictionary revision date when you say so: both codes were added after most
  published NYC taxi analyses were written, and a marker who knows the older dictionary
  will want to see that the decision was made deliberately rather than by omission.
- The Step 4a profile turns the retention decision from an assertion into a finding.
  State how the unrecorded records break down by vendor and pickup zone, and state the
  citywide and airport rates side by side — the gap between the two is the reason the
  decision affects the target less than the citywide figure suggests.
- Two filters remove nothing. That is worth one clause, not a paragraph: every code value
  present in eighteen months of records is one the dictionary defines.
- `MIN_METERED_FARE` is the $3.00 initial unit charge in force from 19 December 2022 (TLC
  Industry Notice #22-02). Cite it wherever the threshold is mentioned; the older $2.50
  figure appears in most published analyses and a reader may otherwise assume it applies.

**Next:** notebook 2b — flight-side aggregation to the same `(date, hour, airport)` key,
then 2c for the join and the temporal, weather, and lag features.